# 05_06 Saving and loading the model

*📖 Book: section 5.4*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/checkpoint.py`
> - `src/homer_ai/train_homer.py`
> - `src/homer_ai/homer_mcp_server.py`
>
> **This notebook:** Training took minutes and the result lives only in memory. How do we save it to a file, load it back exactly, and use it somewhere else, like the MCP server?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What a checkpoint is, and what ours holds (weights, sizes, vocabulary, training notes)
- How to save the weights and load them into a fresh model
- What goes wrong when the shapes differ
- Why `weights_only=True` is safer
- How the MCP server loads our model

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **state_dict** | a Python dictionary: the name of each weight group and its numbers | `model.state_dict()` |
| **pickle** | Python's way of turning objects into bytes; loading a pickle can run code hidden inside it | why `.pt` files from strangers can be unsafe |
| **weights_only** | a `torch.load` setting that only allows plain numbers and text, no hidden code | `weights_only=True` |
| **temporary folder** | a folder Python makes for you and deletes afterwards, so a demo can't touch real files | `tempfile.TemporaryDirectory()` |

Words you've met before: **checkpoint** ([01_04](../01_foundations/01_04_our_model_as_a_black_box.ipynb)).

## 3. The idea

### ❓ Training took minutes. How do I keep the result, and use it later or somewhere else?

When the notebook or script ends, everything in memory is gone, including the trained weights. So at the end of training we write them to a file. That file is the **checkpoint**. Later, another program (like our MCP server in [chapter 6](../06_making_it_talk/06_00_chapter_overview.ipynb)) reads it and gets the trained model back, with no training needed.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ What's in our checkpoint?

Everything you need to use the model without asking anyone. Our files (**format 2**) hold six things:

```
homer_gpt.pt
├─ "format":      2
├─ "kind":        "gpt"  (or "char-1head", the small model)
├─ "config":      the sizes: vocab_size, context_length, emb_dim, n_heads, n_layers, …
├─ "chars":       the vocabulary, in order (the tokenizer's table)
├─ "model_state": the weights (a state_dict)
└─ "train_meta":  how it was trained: seed, epochs, learning rate, losses, books, versions
```

Why the sizes? A bare `state_dict` only fits a model of the same shape (we break that on purpose below). Without the sizes in the file, the server and every notebook would each have to *know* `embed_dim=32`. With them, the file speaks for itself, and one loader serves any model. Here's the real code that writes it (printed with `inspect.getsource`, which shows a function's code).

👉 **Predict first:** which item is biggest on disk?

In [ ]:
import inspect

from homer_ai.checkpoint import save_checkpoint

print(inspect.getsource(save_checkpoint))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

One dictionary, written by `torch.save`. The `state_dict` is by far the biggest part: the whole GPT file is 3.4 MB (835,072 weights), and the small model's is 88 KB. The config, the vocabulary and the notes come to a few kilobytes. `git_sha` is the version of the code (if git is there), and `torch_version` is the PyTorch that wrote it.

</details>

### ❓ Can I look inside a real one?

Yes. We open the ready-made GPT of [models/](../../models/README.md) with `weights_only=True` (more on that below) and print everything except the weights. We also look at the ready-made small model. Its file is in **format 1**, the simpler of our two checkpoint formats, and has only two keys.

In [ ]:
import torch

from homer_ai.lessons import READY_MADE_GPT, READY_MADE_MODEL

gpt_file = torch.load(READY_MADE_GPT, weights_only=True)  # read only
print("keys:", list(gpt_file))
print("kind:", gpt_file["kind"], "| config:", gpt_file["config"])
chars = gpt_file["chars"]
print("vocabulary:", len(chars), "tokens, the last three:", chars[-3:])
for name, value in gpt_file["train_meta"].items():
    print(f"  {name:18}{value}")

small_file = torch.load(READY_MADE_MODEL, weights_only=True)
print("\nsmall model (format 1), keys:", list(small_file))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Six keys. The config is a plain dictionary (context 64, 128 numbers wide, 4 heads, 4 blocks, dropout 0.3). The vocabulary has 137 tokens, ending with `<|unk|>` and `<|endoftext|>`. `train_meta` is the lab notebook: seed 42, 200 epochs, 17,730 letters to train on and 1,971 held out, the held-out loss 2.27, and how long it took. The format-1 file has only `model_state` and `rhapsody_text` (the whole of Book 1 as text). A **rhapsody** is the old Greek name for one book of a Homeric poem, a piece a singer (a *rhapsode*) performed in one go. So `rhapsody_text` just means "the text of the book it trained on". So that's the two formats: format 1 is weights plus text, and format 2 adds the settings, the vocabulary and the notes. `load_checkpoint` reads both.

</details>

### ❓ Why save the vocabulary (the `chars` list)?

Our tokenizer ([02_01](../02_text_to_numbers/02_01_tokens_and_vocabulary.ipynb)) is a table of characters in a fixed order, and the ID of a character is its place in that table. So the same table gives the same IDs. The weights are matched to it: token ID 41 still means `π`, because the model learned with that table. A different table would give a different numbering, and the trained weights would read nonsense. So we save the table itself, and `CharTokenizer.from_chars` rebuilds it. (A format-1 file carries the whole book, and the table is rebuilt from that text. Same result, but a lot more to carry.)

👉 **Predict first:** a tokenizer rebuilt from the saved table: does it give the same IDs as the original?

In [ ]:
from homer_ai.tokenizer import CharTokenizer

sample = "ἄνδρα μοι ἔννεπε"
original = CharTokenizer(sample)
rebuilt = CharTokenizer.from_chars(original.chars)
print(original.encode("ἄνδρα"))
print(rebuilt.encode("ἄνδρα"))
print("same IDs?", original.encode("ἄνδρα") == rebuilt.encode("ἄνδρα"))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same list of numbers twice, so `True`. That's why the table is a safe thing to carry around.

</details>

### ❓ How do I save weights and load them into a fresh model?

`torch.save(object, path)` writes, `torch.load(path)` reads, and `load_state_dict` copies the numbers into a model of the same shape. We do it in a **temporary folder**, so we never touch the real `homer_model.pt`. The test is simple: the original and the copy must give identical outputs. `torch.equal` says whether two tensors are exactly the same.

👉 **Predict first:** a brand-new model has random weights. Does it give the same output as the saved one before loading? After?

In [ ]:
import tempfile
from pathlib import Path

import torch

from homer_ai.model import HomerGreekLLM

ids = torch.tensor([3, 1, 4, 1, 5, 9, 2, 6])  # a few made-up character IDs

torch.manual_seed(1)
original = HomerGreekLLM(vocab_size=10, embed_dim=32)

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "small.pt"
    torch.save({"model_state": original.state_dict()}, path)
    print("file size:", path.stat().st_size, "bytes")

    torch.manual_seed(2)  # different random start
    fresh = HomerGreekLLM(vocab_size=10, embed_dim=32)
    with torch.no_grad():
        print("before loading, same output?", torch.equal(original(ids), fresh(ids)))
        fresh.load_state_dict(torch.load(path, weights_only=True)["model_state"])
        print("after loading, same output? ", torch.equal(original(ids), fresh(ids)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`False` before, `True` after. Once loaded, the fresh model **is** the saved one, down to the last digit. The temporary folder, and the file in it, are gone after the `with` block.

</details>

### ❓ What goes wrong if the new model has a different shape?

`load_state_dict` refuses, because the saved numbers don't fit. [A_07](../A_toolbox/A_07_organising_code.ipynb) shows the same error, and it's why `model.py` is shared.

👉 **Predict first:** the file holds weights for `embed_dim=32`. Load them into a model with `embed_dim=16`.

In [ ]:
small = HomerGreekLLM(vocab_size=10, embed_dim=16)
try:
    small.load_state_dict(original.state_dict())
except RuntimeError as error:
    print("Could not load:", str(error).splitlines()[1].strip())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A size-mismatch error that names the first layer that doesn't fit. A loud error is a good thing here. PyTorch never quietly accepts wrong shapes. So the model you build must have the **same settings** (`vocab_size`, `embed_dim`) as the one you saved. That's why a format-2 checkpoint carries its `config`, and `load_checkpoint` builds the model from it.

</details>

### ❓ What's `weights_only=True`, and why does the server use it?

A `.pt` file is a **pickle**, and loading a pickle can run any code hidden inside it. So a `.pt` file from a stranger could harm your computer. `weights_only=True` tells `torch.load` to accept only plain numbers, text and dictionaries, and to refuse anything else.

Our checkpoint is just a dictionary of tensors and text, so it loads fine with the flag on. A rule worth keeping: always set it, unless you trust the file completely.

### ❓ How does the MCP server load our model?

It calls `load_checkpoint` (in `checkpoint.py`), which does what we did above, in a few lines. Let's print them from the file (we read it as text, so there's no need to import it).

👉 **Predict first:** which lines match what we just did by hand?

In [ ]:
from homer_ai.train_homer import HERE

checkpoint_file = HERE / "checkpoint.py"  # we only read it as text, never import it
keys = ("torch.load", "from_chars(", "= build_model", "load_state_dict", ".eval()")
for line in checkpoint_file.read_text(encoding="utf-8").splitlines():
    if any(key in line for key in keys):
        print(line.strip())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`torch.load(..., weights_only=True)`, the tokenizer rebuilt from the saved table (`from_chars`; for a format-1 file, from its text), a fresh model of the right kind and size (`build_model`), then `load_state_dict`. Then comes `model.eval()`, which [chapter 6](../06_making_it_talk/06_00_chapter_overview.ipynb) explains. So the server needs no training and no numbers of its own. Just this file.

</details>

### ❓ Can I save the optimizer too, to continue training later?

Yes. The optimizer (`AdamW`) keeps its own memory of recent steps, and that memory isn't in the weights. Add its `state_dict()` to the checkpoint, and training can carry on where it stopped. Our checkpoints don't have it, because we only need to use the model, not train it further.

In [ ]:
import torch.nn.functional as F

learner = HomerGreekLLM(vocab_size=10, embed_dim=32)  # separate: `original` stays
optimizer = torch.optim.AdamW(learner.parameters(), lr=0.01)
loss = F.cross_entropy(learner(ids[:-1]), ids[1:])  # guess each next ID
loss.backward()
optimizer.step()  # one step, so the optimizer has something to remember

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "resume.pt"
    torch.save(
        {
            "model_state": learner.state_dict(),
            "optimizer_state": optimizer.state_dict(),
        },
        path,
    )
    saved = torch.load(path, weights_only=True)
    print("keys in the file:", list(saved))
    print("optimizer remembers:", list(saved["optimizer_state"]["state"][0]))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A second key, `optimizer_state`, which holds the optimizer's memory (`exp_avg` is a running average of each weight's recent nudges, `exp_avg_sq` a running average of their squares, and AdamW uses both to size the next nudge). It loads with `weights_only=True` too.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why not save the whole model object with `torch.save(model, path)`?

It works, but it ties the file to the exact code layout that made it, and loading it needs full pickle trust. Saving the `state_dict` (plain numbers) is safer and more flexible.

## 5. Recap

- A checkpoint (format 2) = the weights (`state_dict`) + the config (sizes) + the vocabulary + notes on how it was trained. The same vocabulary gives the same IDs.
- Save with `torch.save`, load with `load_state_dict` into a model of the same shape. `torch.equal` proves the copy is exact.
- Load with `weights_only=True`. The MCP server does just that.

**You can now:**

- save a model with its vocabulary and settings, and load it back
- prove the loaded copy is exact with `torch.equal`

**Next:** [05_07: Train the GPT](05_07_train_the_gpt.ipynb): how the GPT in that checkpoint was trained.